# FiberNet v4.0 Tutorial — 从生成到优化的完整流水线

## Complete Pipeline: Structure Generation → Simulation → Analysis → ML → RL

本教程演示完整工作流：
1. 生成 12 种基础结构类型 + 蜂巢基(honeycomb)变体
2. 机械模拟（拉伸测试）
3. 结构特征提取（94维）
4. 机器学习预测力学性能
5. 强化学习优化结构参数

**关键参数**：
- `perturbation`: 中间点位移幅度，以**平均边长百分比**计算
- 教程用 `perturbation=0.20`（20%），生产用 `0.40`（40%）

## 1. Setup / 环境设置

In [ ]:
import os, sys, json, time, warnings, gc, copy
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── Output directories ──
DATA_OUT = Path('tutorial_data')
VIZ_OUT = Path('tutorial_viz')
DATA_OUT.mkdir(exist_ok=True)
VIZ_OUT.mkdir(exist_ok=True)

print(f'✓ Output directories:')
print(f'  Data: {DATA_OUT.resolve()}')
print(f'  Viz:  {VIZ_OUT.resolve()}')

## 2. Import FiberNet / 导入验证

In [ ]:
import fibernet as fn
from fibernet import pattern_2d, TaichiEngine, list_units
from fibernet.viz.render import render_graph, _get_theme
from fibernet.sim.accelerated import _graph_to_arrays
from fibernet.analysis import GraphFeatureExtractor
from fibernet.rl import ParametricStructureEnv

print(f'✓ FiberNet loaded')
print(f'  Available units ({len(list_units())}): {list_units()}')

## 3. Structure Generation / 结构生成

### 3.1 12 种基础单元类型

FiberNet 提供 12 种基础单元类型，每种都可以平铺(tiling)为更大的结构。

In [ ]:
# ── Base parameters ──
BOX = (1.0, 1.0)
GRID = (4, 4)
units = list_units()  # 12 types

base_structures = {}
print('Generating 12 base unit types (grid=4×4):')
for unit in units:
    g = pattern_2d(unit=unit, box=BOX, grid=GRID)
    base_structures[unit] = g
    print(f'  {unit:12s}: {g.num_nodes:4d} nodes, {g.num_edges:4d} edges')

# Save gallery JSON
gallery_data = []
for unit, g in base_structures.items():
    pos, elems, nids, _ = _graph_to_arrays(g)
    gallery_data.append({
        'unit': unit,
        'num_nodes': g.num_nodes,
        'num_edges': g.num_edges,
        'positions': pos[:, :2].tolist(),
        'elements': elems.tolist(),
    })

with open(DATA_OUT / 'base_structures_gallery.json', 'w') as f:
    json.dump(gallery_data, f, indent=2)

print(f'\n✓ Generated {len(base_structures)} base structures')
print(f'  Saved: base_structures_gallery.json')

### 3.1.1 Gallery — 12 Base Unit Types (暗色+亮色)

In [ ]:
# ── Render 12 base structures gallery ──
for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    fig.patch.set_facecolor(colors['bg'])
    
    for ax, unit in zip(axes.flat, units):
        g = base_structures[unit]
        render_graph(g, ax=ax, theme=theme, color_by='uniform',
                     line_width=1.5, show_nodes=False)
        ax.set_title(unit.replace('_', ' ').title(),
                     color=colors['text'], fontsize=12, fontweight='bold')
    
    fig.suptitle('12 Base Unit Types (Undeformed, 4×4 grid)',
                 color=colors['text'], fontsize=16, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'01_gallery_undeformed_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')

print('\n✓ Gallery saved (dark + light)')

### 3.2 Intermediate Point Deformations (中间点变形)

每条边可以添加中间点来创建弯曲/波浪形的梁。

**关键参数**：
- `n_pts_per_side`: 每条边的中间点数
- `perturbation`: 位移幅度 = **平均边长 × 百分比**

例如：`perturbation=0.20` 意味着位移 = 平均边长 × 20%

In [ ]:
# ── Perturbation comparison: 0% to 80% ──
perturbations = [0.0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.80]
N_PTS = 3  # 3 intermediate points per edge

for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, axes = plt.subplots(2, 4, figsize=(24, 12))
    fig.patch.set_facecolor(colors['bg'])
    
    for ax, pert in zip(axes.flat, perturbations):
        if pert == 0.0:
            g = pattern_2d(unit='honeycomb', box=BOX, grid=GRID,
                           seed=42, n_pts_per_side=0)
        else:
            g = pattern_2d(unit='honeycomb', box=BOX, grid=GRID,
                           seed=42, n_pts_per_side=N_PTS, perturbation=pert)
        render_graph(g, ax=ax, theme=theme, color_by='uniform',
                     line_width=1.2, show_nodes=False)
        label = f'perturbation={pert:.2f}\n({pert*100:.0f}% of edge length)'
        if pert == 0.0:
            label = 'perturbation=0.00\n(no intermediate points)'
        ax.set_title(label, color=colors['text'], fontsize=10)
    
    fig.suptitle('Intermediate Point Deformations (Honeycomb, n_pts_per_side=3)',
                 color=colors['text'], fontsize=15, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'02_perturbation_comparison_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')

print('\n✓ Perturbation comparison saved (dark + light)')

### 3.3 12 种基础单元 + 中间点变形

所有 12 种基础单元类型加上中间点变形（`n_pts_per_side=3, perturbation=0.20`）。

In [ ]:
# ── Generate deformed versions of all 12 units ──
deformed_structures = {}
print('Generating deformed structures (n_pts_per_side=3, perturbation=0.20):')
for unit in units:
    g = pattern_2d(unit=unit, box=BOX, grid=GRID,
                   n_pts_per_side=3, perturbation=0.20, seed=42)
    deformed_structures[unit] = g
    print(f'  {unit:12s}: {g.num_nodes:4d} nodes, {g.num_edges:4d} edges')

# Save deformed gallery JSON
deformed_data = []
for unit, g in deformed_structures.items():
    pos, elems, nids, _ = _graph_to_arrays(g)
    deformed_data.append({
        'unit': unit,
        'num_nodes': g.num_nodes,
        'num_edges': g.num_edges,
        'n_pts_per_side': 3,
        'perturbation': 0.20,
        'positions': pos[:, :2].tolist(),
        'elements': elems.tolist(),
    })

with open(DATA_OUT / 'deformed_structures_gallery.json', 'w') as f:
    json.dump(deformed_data, f, indent=2)
print(f'\n✓ Saved: deformed_structures_gallery.json')

In [ ]:
# ── Render deformed gallery ──
for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    fig.patch.set_facecolor(colors['bg'])
    
    for ax, unit in zip(axes.flat, units):
        g = deformed_structures[unit]
        render_graph(g, ax=ax, theme=theme, color_by='uniform',
                     line_width=1.2, show_nodes=False)
        ax.set_title(f"{unit.replace('_', ' ').title()}\n(n_pts=3, pert=20%)",
                     color=colors['text'], fontsize=10)
    
    fig.suptitle('12 Base Unit Types (With Intermediate Point Deformations)',
                 color=colors['text'], fontsize=14, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'03_gallery_deformed_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')

print('\n✓ Deformed gallery saved (dark + light)')

### 3.4 Batch Generation: Honeycomb Variants

生成大量蜂巢(honeycomb)结构的变体，每个有不同的随机种子和中间点变形。

**教程用 20 个做演示，生产用 2000 个。**

In [ ]:
# ── Batch Generation Parameters ──
UNIT = 'honeycomb'
N_PTS_PER_SIDE = 3
PERTURBATION = 0.20    # Tutorial: 20% of edge length
# PERTURBATION = 0.40  # Production: 40% of edge length (uncomment for production)

N_STRUCTURES = 20      # Tutorial: 20 structures
# N_STRUCTURES = 2000  # Production: 2000 structures (uncomment for production)

print('Batch generation parameters:')
print(f'  UNIT:             {UNIT}')
print(f'  N_PTS_PER_SIDE:   {N_PTS_PER_SIDE}')
print(f'  PERTURBATION:     {PERTURBATION} ({PERTURBATION*100:.0f}% of edge length)')
print(f'  N_STRUCTURES:     {N_STRUCTURES}')
print(f'  GRID:             {GRID}')
print()

# Generate structures
all_structures = []
all_metadata = []

for i in tqdm(range(N_STRUCTURES), desc='Generating'):
    g = pattern_2d(
        unit=UNIT, box=BOX, grid=GRID, seed=100+i,
        n_pts_per_side=N_PTS_PER_SIDE, perturbation=PERTURBATION
    )
    all_structures.append(g)
    all_metadata.append({
        'name': f'honeycomb_{i:03d}',
        'seed': 100+i,
        'nodes': g.num_nodes,
        'edges': g.num_edges,
        'perturbation': PERTURBATION,
    })

print(f'\n✓ Generated {N_STRUCTURES} honeycomb structures')
print(f'  Nodes: {all_metadata[0]["nodes"]} each')
print(f'  Edges: {all_metadata[0]["edges"]} each')

### 3.5 Gallery — Random Selection of Generated Structures

In [ ]:
# ── Visualize random 20 structures from the pool ──
# For tutorial we show all 20; for production (2000) we randomly sample 20
import random

if len(all_structures) <= 20:
    show_structures = all_structures
    show_metadata = all_metadata
    n_show = len(show_structures)
else:
    indices = sorted(random.sample(range(len(all_structures)), 20))
    show_structures = [all_structures[i] for i in indices]
    show_metadata = [all_metadata[i] for i in indices]
    n_show = 20

n_cols = 5
n_rows = (n_show + n_cols - 1) // n_cols

for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4*n_rows))
    fig.patch.set_facecolor(colors['bg'])
    
    for idx, (ax, g, meta) in enumerate(zip(axes.flat, show_structures, show_metadata)):
        render_graph(g, ax=ax, theme=theme, color_by='uniform',
                     line_width=1.0, show_nodes=False)
        ax.set_title(f'{meta["name"]} (seed={meta["seed"]})',
                     color=colors['text'], fontsize=9)
    
    # Hide empty axes
    for ax in axes.flat[n_show:]:
        ax.axis('off')
    
    title = f'{n_show} Honeycomb Variants (perturbation={PERTURBATION})'
    if N_STRUCTURES > 20:
        title += f'\n(Random sample from {N_STRUCTURES} total)'
    fig.suptitle(title, color=colors['text'], fontsize=14, fontweight='bold', y=1.0)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'04_gallery_batch_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')

print('\n✓ Batch gallery saved (dark + light)')

## 4. Simulation / 模拟

对所有结构进行拉伸测试。

**API 方法**: `TaichiEngine.stretch_test()`

**模拟参数**：
- `stiffness=1e5`: 高刚度确保波传播
- `damping=0.3`: 低阻尼减少能量损失
- `num_steps=15000`: 7500 步爬坡 + 7500 步保持
- `target_stretch=1.5`: 50% 拉伸
- `ramp_fraction=0.5`: 50% 时间用于爬坡，50% 弛豫

In [ ]:
# ── Simulation Parameters ──
STIFFNESS = 1e5
DAMPING = 0.3
NUM_STEPS = 15000
RAMP_FRACTION = 0.5
TARGET_STRETCH = 1.5

print('Simulation parameters:')
print(f'  STIFFNESS:       {STIFFNESS:.1e} N/m')
print(f'  DAMPING:         {DAMPING}')
print(f'  NUM_STEPS:       {NUM_STEPS}')
print(f'  RAMP_FRACTION:   {RAMP_FRACTION}')
print(f'  TARGET_STRETCH:  {TARGET_STRETCH}x')
print()

engine = TaichiEngine()
sim_results = []

# Checkpoint support
ckpt_path = DATA_OUT / 'sim_checkpoint.json'
start_idx = 0
if ckpt_path.exists():
    with open(ckpt_path) as f:
        ckpt = json.load(f)
    start_idx = len(ckpt)
    sim_results = ckpt
    print(f'Resuming from checkpoint: {start_idx}/{N_STRUCTURES}')

for i in tqdm(range(start_idx, N_STRUCTURES), desc='Simulating'):
    g = all_structures[i]
    result = engine.stretch_test(
        g,
        target_stretch=TARGET_STRETCH,
        stiffness=STIFFNESS,
        damping=DAMPING,
        num_steps=NUM_STEPS,
        ramp_fraction=RAMP_FRACTION,
        save_interval=500
    )
    
    sim_dict = result.to_dict()
    sim_dict['index'] = i
    sim_dict['name'] = all_metadata[i]['name']
    sim_results.append(sim_dict)
    
    # Save individual result
    result.save(str(DATA_OUT / f'{all_metadata[i]["name"]}_sim.json'), detailed=True)
    
    # Checkpoint every 5 structures
    if (i + 1) % 5 == 0:
        with open(ckpt_path, 'w') as f:
            json.dump(sim_results, f)
        gc.collect()

# Save final checkpoint
with open(ckpt_path, 'w') as f:
    json.dump(sim_results, f)

# Extract metrics
for meta, sr in zip(all_metadata, sim_results):
    meta['max_force'] = sr['max_force'] if isinstance(sr, dict) else sr.max_force
    meta['max_stretch'] = sr['max_stretch'] if isinstance(sr, dict) else sr.max_stretch
    meta['energy'] = sr['energy'] if isinstance(sr, dict) else sr.energy

forces = [m['max_force'] for m in all_metadata]
print(f'\n✓ Completed {len(sim_results)} simulations')
print(f'  Force range: {min(forces):.0f} - {max(forces):.0f} N')
print(f'  Mean force:  {np.mean(forces):.0f} N')

### 4.2 Deformation Trajectory (变形轨迹)

展示单个结构的 8 帧变形过程。

**注意**: `render_graph` 不支持直接传入变形后坐标，
这里使用 `_graph_to_arrays` 获取边信息后用 `LineCollection` 绘制。

In [ ]:
def draw_deformed_structure(g, sim_result_or_dict, ax, colors, color_by_stretch=False):
    """Draw a structure in its deformed state.
    
    Parameters
    ----------
    g : StructureGraph
    sim_result_or_dict : SimResult or dict
        Must have .deformed_positions or ['deformed_positions']
    ax : matplotlib Axes
    colors : dict with 'bg', 'fiber', 'grid', 'text'
    color_by_stretch : bool
        If True, color edges by stretch ratio
    """
    if isinstance(sim_result_or_dict, dict):
        pos_def = np.array(sim_result_or_dict['deformed_positions'])
    else:
        pos_def = sim_result_or_dict.deformed_positions
    
    pos_orig, elements, node_ids, _ = _graph_to_arrays(g)
    
    if color_by_stretch:
        lengths_orig = np.array([np.linalg.norm(
            pos_orig[elements[e,1]] - pos_orig[elements[e,0]]) for e in range(len(elements))])
        lengths_def = np.array([np.linalg.norm(
            pos_def[elements[e,1]] - pos_def[elements[e,0]]) for e in range(len(elements))])
        stretch = lengths_def / (lengths_orig + 1e-12)
        
        norm = Normalize(vmin=stretch.min(), vmax=stretch.max())
        cmap = plt.cm.RdYlGn_r
        
        segments = []
        colors_list = []
        for e_idx, e in enumerate(elements):
            p0 = pos_def[e[0]]
            p1 = pos_def[e[1]]
            segments.append([[p0[0], p0[1]], [p1[0], p1[1]]])
            colors_list.append(cmap(norm(stretch[e_idx])))
        
        lc = LineCollection(segments, colors=colors_list, linewidths=1.5, capstyle='round')
        ax.add_collection(lc)
        ax.set_aspect('equal')
        ax.autoscale()
        return stretch
    else:
        for e in elements:
            ax.plot([pos_def[e[0],0], pos_def[e[1],0]],
                    [pos_def[e[0],1], pos_def[e[1],1]],
                    color=colors['fiber'], linewidth=1.2, alpha=0.8)
        ax.set_aspect('equal')
        ax.autoscale()
        return None

print('✓ Helper function defined: draw_deformed_structure()')

In [ ]:
# ── Trajectory visualization (8 frames) ──
# Re-run simulation for structure 0 to get trajectory
g0 = all_structures[0]
result0 = engine.stretch_test(
    g0, target_stretch=TARGET_STRETCH, stiffness=STIFFNESS,
    damping=DAMPING, num_steps=NUM_STEPS, ramp_fraction=RAMP_FRACTION,
    save_interval=500
)

print(f'Structure 0: max_force={result0.max_force:.0f} N, '
      f'max_stretch={result0.max_stretch:.3f}')

traj = result0.positions_trajectory
if traj is None:
    # Fallback: just use initial + final
    pos_orig, _, _, _ = _graph_to_arrays(g0)
    traj = [pos_orig, result0.deformed_positions]

print(f'Trajectory frames: {len(traj)}')

for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    
    n_frames = min(8, len(traj))
    frame_indices = np.linspace(0, len(traj)-1, n_frames, dtype=int)
    
    fig, axes = plt.subplots(2, 4, figsize=(24, 12))
    fig.patch.set_facecolor(colors['bg'])
    axes = axes.flatten()
    
    pos_orig, elements, node_ids, _ = _graph_to_arrays(g0)
    
    for idx, fi in enumerate(frame_indices):
        ax = axes[idx]
        ax.set_facecolor(colors['bg'])
        
        pos_frame = np.array(traj[fi])
        for e in elements:
            ax.plot([pos_frame[e[0],0], pos_frame[e[1],0]],
                    [pos_frame[e[0],1], pos_frame[e[1],1]],
                    color=colors['fiber'], linewidth=1.0, alpha=0.8)
        
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(f'Frame {fi}/{len(traj)-1}',
                     color=colors['text'], fontsize=10)
    
    fig.suptitle(f'Deformation Trajectory: {all_metadata[0]["name"]} (8 frames)',
                 color=colors['text'], fontsize=15, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'05_trajectory_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')

print('\n✓ Trajectory saved (dark + light)')

### 4.3 Stress Distribution (应力分布)

In [ ]:
# ── Stress distribution: original vs deformed ──
for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 9))
    fig.patch.set_facecolor(colors['bg'])
    
    # Left: original
    render_graph(g0, ax=ax1, theme=theme, color_by='uniform',
                 line_width=1.5, show_nodes=False)
    ax1.set_title('Original', color=colors['text'], fontsize=14)
    
    # Right: deformed with stress coloring
    ax2.set_facecolor(colors['bg'])
    stretch = draw_deformed_structure(g0, result0, ax2, colors, color_by_stretch=True)
    ax2.tick_params(colors=colors['text'])
    for spine in ax2.spines.values():
        spine.set_color(colors['grid'])
    
    # Colorbar
    norm = Normalize(vmin=stretch.min(), vmax=stretch.max())
    sm = ScalarMappable(cmap=plt.cm.RdYlGn_r, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax2, fraction=0.046, pad=0.04)
    cbar.set_label('Stretch Ratio', color=colors['text'])
    cbar.ax.tick_params(colors=colors['text'])
    
    ax2.set_title(f'Deformed (Stretch: {stretch.min():.2f}-{stretch.max():.2f})',
                  color=colors['text'], fontsize=14)
    
    fig.suptitle(f'Stress Distribution: {all_metadata[0]["name"]}',
                 color=colors['text'], fontsize=15, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'06_stress_distribution_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')

print('\n✓ Stress distribution saved (dark + light)')

### 4.4 Batch Simulation Statistics

In [ ]:
# ── Batch statistics ──
forces = [m['max_force'] for m in all_metadata]
stretches = [m['max_stretch'] for m in all_metadata]
energies = [m['energy'] for m in all_metadata]

for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    fig.patch.set_facecolor(colors['bg'])
    
    for ax in axes.flat:
        ax.set_facecolor(colors['bg'])
        ax.tick_params(colors=colors['text'])
        for spine in ax.spines.values():
            spine.set_color(colors['grid'])
        ax.xaxis.label.set_color(colors['text'])
        ax.yaxis.label.set_color(colors['text'])
        ax.title.set_color(colors['text'])
    
    # Force distribution
    ax = axes[0, 0]
    ax.hist(forces, bins=max(5, N_STRUCTURES//2), color=colors['fiber'],
            alpha=0.7, edgecolor=colors['grid'])
    ax.axvline(np.mean(forces), color='red', linestyle='--', linewidth=2)
    ax.set_xlabel('Max Force (N)')
    ax.set_ylabel('Count')
    ax.set_title(f'Force Distribution\nMean={np.mean(forces):.0f} N')
    
    # Force by structure
    ax = axes[0, 1]
    ax.plot(range(len(forces)), forces, 'o-', color=colors['fiber'],
            linewidth=2, markersize=6)
    ax.set_xlabel('Structure Index')
    ax.set_ylabel('Max Force (N)')
    ax.set_title('Force by Structure')
    ax.grid(True, alpha=0.3, color=colors['grid'])
    
    # Energy distribution
    ax = axes[1, 0]
    ax.hist(energies, bins=max(5, N_STRUCTURES//2), color=colors.get('accent', colors['fiber']),
            alpha=0.7, edgecolor=colors['grid'])
    ax.set_xlabel('Energy')
    ax.set_ylabel('Count')
    ax.set_title(f'Energy Distribution\nMean={np.mean(energies):.0f}')
    
    # Stretch distribution
    ax = axes[1, 1]
    ax.hist(stretches, bins=max(5, N_STRUCTURES//2), color=colors['fiber'],
            alpha=0.7, edgecolor=colors['grid'])
    ax.set_xlabel('Max Stretch Ratio')
    ax.set_ylabel('Count')
    ax.set_title(f'Stretch Distribution\nMean={np.mean(stretches):.3f}')
    
    fig.suptitle(f'Batch Simulation Statistics ({N_STRUCTURES} structures)',
                 color=colors['text'], fontsize=14, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'07_batch_stats_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')

print('\n✓ Batch statistics saved (dark + light)')

## 5. Feature Extraction / 特征提取

提取 94 维结构特征用于 ML/RL：
- **几何特征**: 节点数、边数、总长度、平均边长
- **拓扑特征**: 度分布、聚类系数、度熵
- **方向特征**: 取向熵、各向异性

In [ ]:
extractor = GraphFeatureExtractor()

for i, (g, meta) in enumerate(tqdm(list(zip(all_structures, all_metadata)),
                                     desc='Extracting features')):
    features = extractor.extract(g)
    meta.update(features)

feature_keys = [k for k in all_metadata[0].keys() if k not in
                ['name', 'seed', 'nodes', 'edges', 'max_force', 'max_stretch',
                 'energy', 'perturbation']]

print(f'✓ Extracted {len(feature_keys)} features per structure')
print(f'\nSample features:')
for k in feature_keys[:10]:
    vals = [m[k] for m in all_metadata]
    print(f'  {k:25s}: min={min(vals):.4f}, mean={np.mean(vals):.4f}, max={max(vals):.4f}')

# Remove features with zero variance
valid_features = []
for k in feature_keys:
    vals = [m[k] for m in all_metadata]
    if np.std(vals) > 1e-12 and not all(v == 0 for v in vals):
        valid_features.append(k)

print(f'\nValid features: {len(valid_features)} / {len(feature_keys)}')

# Save to CSV
import pandas as pd
df = pd.DataFrame(all_metadata)
df.to_csv(DATA_OUT / 'structures_features.csv', index=False)
print(f'✓ Saved: structures_features.csv ({len(df)} rows, {len(df.columns)} columns)')

### 5.1 Feature Statistics

In [ ]:
# ── Feature statistics visualization ──
# Show top 20 features by variance
variances = {}
for k in valid_features:
    vals = [m[k] for m in all_metadata]
    variances[k] = np.var(vals)

top_features = sorted(variances, key=variances.get, reverse=True)[:20]

for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, axes = plt.subplots(4, 5, figsize=(20, 16))
    fig.patch.set_facecolor(colors['bg'])
    
    for i, (feat, ax) in enumerate(zip(top_features, axes.flat)):
        ax.set_facecolor(colors['bg'])
        ax.tick_params(colors=colors['text'])
        for spine in ax.spines.values():
            spine.set_color(colors['grid'])
        
        vals = [m[feat] for m in all_metadata]
        ax.hist(vals, bins=max(5, N_STRUCTURES//3), color=colors['fiber'],
                alpha=0.7, edgecolor=colors['grid'])
        ax.set_title(feat.replace('_', '\n'), color=colors['text'], fontsize=8)
        ax.set_ylabel('Count', fontsize=8)
    
    for ax in axes.flat[len(top_features):]:
        ax.axis('off')
    
    fig.suptitle(f'Top 20 Features by Variance ({len(valid_features)} valid / {len(feature_keys)} total)',
                 color=colors['text'], fontsize=14, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'08_feature_stats_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')

print('\n✓ Feature statistics saved (dark + light)')

## 6. Machine Learning / 机器学习

训练 Random Forest 模型预测力学性能。

**注意**: 教程中只有 20 个样本，ML 效果有限。
生产环境使用 2000 个样本效果更好。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_squared_error, confusion_matrix, accuracy_score

# Prepare data
X = np.array([[m[k] for k in valid_features] for m in all_metadata])
y = np.array([m['max_force'] for m in all_metadata])

# Split
if N_STRUCTURES >= 10:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)
else:
    X_train, X_test, y_train, y_test = X, X, y, y

# Train regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=8,
                                oob_score=True)
rf_reg.fit(X_train, y_train)
y_pred = rf_reg.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# Train classifier (high vs low force)
threshold = np.median(y)
y_train_bin = (y_train > threshold).astype(int)
y_test_bin = (y_test > threshold).astype(int)
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train_bin)
y_pred_bin = rf_clf.predict(X_test)
cm = confusion_matrix(y_test_bin, y_pred_bin)
acc = accuracy_score(y_test_bin, y_pred_bin)

print(f'✓ Random Forest trained')
print(f'  R² Score:    {r2:.3f}')
print(f'  RMSE:        {rmse:.0f} N')
print(f'  OOB Score:   {rf_reg.oob_score_:.3f}')
print(f'  Accuracy:    {acc:.3f} (threshold={threshold:.0f} N)')

# Feature importance
importances = rf_reg.feature_importances_
top_idx = np.argsort(importances)[-15:][::-1]

print(f'\nTop 15 features:')
for i in top_idx:
    print(f'  {valid_features[i]:30s}: {importances[i]:.3f}')

In [ ]:
# ── ML Visualization ──
for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    fig.patch.set_facecolor(colors['bg'])
    
    for ax in axes.flat:
        ax.set_facecolor(colors['bg'])
        ax.tick_params(colors=colors['text'])
        for spine in ax.spines.values():
            spine.set_color(colors['grid'])
        ax.xaxis.label.set_color(colors['text'])
        ax.yaxis.label.set_color(colors['text'])
        ax.title.set_color(colors['text'])
    
    # Predictions vs Actual
    ax = axes[0, 0]
    ax.scatter(y_test, y_pred, color=colors['fiber'], alpha=0.7, s=60)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
            'r--', linewidth=2, alpha=0.5)
    ax.set_xlabel('Actual Force (N)')
    ax.set_ylabel('Predicted Force (N)')
    ax.set_title(f'Predictions vs Actual\nR²={r2:.3f}, RMSE={rmse:.0f} N')
    
    # Feature importance
    ax = axes[0, 1]
    display_names = [valid_features[i].replace('_', ' ').title() for i in top_idx]
    ax.barh(range(15), importances[top_idx], color=colors['fiber'], alpha=0.7)
    ax.set_yticks(range(15))
    ax.set_yticklabels(display_names, fontsize=8)
    ax.set_xlabel('Importance')
    ax.set_title('Top 15 Feature Importances')
    ax.invert_yaxis()
    
    # Confusion matrix
    ax = axes[1, 0]
    im = ax.imshow(cm, cmap='viridis', interpolation='nearest')
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Low', 'High'])
    ax.set_yticklabels(['Low', 'High'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix\nAcc={acc:.2f}')
    for ci in range(2):
        for cj in range(2):
            tc = 'white' if cm[ci,cj] < cm.max()/2 else 'black'
            ax.text(cj, ci, str(cm[ci, cj]), ha='center', va='center',
                    color=tc, fontsize=20, fontweight='bold')
    
    # OOB loss curve
    ax = axes[1, 1]
    n_est_range = [10, 20, 50, 100, 200]
    oob_errors = []
    for n_est in n_est_range:
        rf_tmp = RandomForestRegressor(n_estimators=n_est, random_state=42,
                                        oob_score=True, max_depth=8)
        rf_tmp.fit(X_train, y_train)
        oob_errors.append(1 - rf_tmp.oob_score_)
    ax.plot(n_est_range, oob_errors, 'o-', color=colors['fiber'],
            linewidth=2, markersize=8)
    ax.set_xlabel('Number of Trees')
    ax.set_ylabel('OOB Error (1 - R²)')
    ax.set_title('Model Complexity vs Error')
    ax.grid(True, alpha=0.3, color=colors['grid'])
    
    fig.suptitle('ML Analysis: Force Prediction', color=colors['text'],
                 fontsize=15, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'09_ml_analysis_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')

print('\n✓ ML analysis saved (dark + light)')

### 6.3 Force-Feature Correlation

In [ ]:
# ── Force-feature correlation ──
corr = {}
y_series = pd.Series([m['max_force'] for m in all_metadata])
for k in valid_features:
    vals = [m[k] for m in all_metadata]
    corr[k] = abs(pd.Series(vals).corr(y_series))

corr_sorted = sorted(corr.items(), key=lambda x: x[1], reverse=True)
top_corr = corr_sorted[:15]

for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    fig.patch.set_facecolor(colors['bg'])
    
    for ax in [ax1, ax2]:
        ax.set_facecolor(colors['bg'])
        ax.tick_params(colors=colors['text'])
        for spine in ax.spines.values():
            spine.set_color(colors['grid'])
        ax.xaxis.label.set_color(colors['text'])
        ax.yaxis.label.set_color(colors['text'])
        ax.title.set_color(colors['text'])
    
    # Top correlations
    ax1.barh(range(15), [c for _, c in top_corr], color=colors['fiber'], alpha=0.7)
    ax1.set_yticks(range(15))
    ax1.set_yticklabels([k.replace('_', ' ').title() for k, _ in top_corr], fontsize=8)
    ax1.set_xlabel('|Correlation|')
    ax1.set_title('Top 15 Force-Feature Correlations')
    ax1.invert_yaxis()
    
    # Top feature scatter
    top_feat = top_corr[0][0]
    vals = [m[top_feat] for m in all_metadata]
    forces = [m['max_force'] for m in all_metadata]
    ax2.scatter(vals, forces, color=colors['fiber'], alpha=0.7, s=60)
    ax2.set_xlabel(top_feat.replace('_', ' ').title())
    ax2.set_ylabel('Max Force (N)')
    ax2.set_title(f'Top Feature: {top_feat}\nCorr={top_corr[0][1]:.3f}')
    ax2.grid(True, alpha=0.3, color=colors['grid'])
    
    fig.suptitle('Force-Feature Importance Analysis', color=colors['text'],
                 fontsize=14, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'10_force_feature_importance_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')

print('\n✓ Correlation analysis saved (dark + light)')

## 7. Reinforcement Learning / 强化学习

使用 RL 优化结构参数以最小化/最大化力学响应。

**RL 环境**: `ParametricStructureEnv`
- 状态空间: 10 维（结构特征）
- 动作空间: 中间点位移（n_sides × n_pts_per_side × 2）
- 奖励: -max_force（最小化力）

In [ ]:
# ── RL Environment ──
env = ParametricStructureEnv(
    unit=UNIT,
    box=BOX,
    grid=GRID,
    n_pts_per_side=N_PTS_PER_SIDE,
    stiffness=STIFFNESS,
    damping=DAMPING,
    num_steps=5000,           # Fewer steps for faster RL
    target_stretch=TARGET_STRETCH,
    reward_mode='minimize_force'
)

print(f'✓ RL Environment created')
print(f'  Unit:        {env.unit}')
print(f'  Grid:        {env.grid}')
print(f'  n_actions:   {env.n_actions}')
print(f'  State dim:   10')
print(f'  Reward mode: {env.reward_mode}')

In [ ]:
# ── RL Training (random search for demo) ──
N_EPISODES = 50
rewards_history = []

print(f'Running {N_EPISODES} RL episodes...')
for ep in tqdm(range(N_EPISODES), desc='RL training'):
    obs = env.reset()
    
    # Random action
    action = np.random.uniform(-0.3, 0.3, size=env.n_actions)
    graph, sim_result, reward, info = env.step(action)
    rewards_history.append(reward)
    
    if (ep + 1) % 10 == 0:
        gc.collect()

print(f'\n✓ RL training complete')
print(f'  Reward range: {min(rewards_history):.0f} - {max(rewards_history):.0f}')
print(f'  Mean reward:  {np.mean(rewards_history):.0f}')

In [ ]:
# ── RL Visualization ──
for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor(colors['bg'])
    
    for ax in [ax1, ax2]:
        ax.set_facecolor(colors['bg'])
        ax.tick_params(colors=colors['text'])
        for spine in ax.spines.values():
            spine.set_color(colors['grid'])
        ax.xaxis.label.set_color(colors['text'])
        ax.yaxis.label.set_color(colors['text'])
        ax.title.set_color(colors['text'])
    
    # Reward curve
    episodes = np.arange(len(rewards_history))
    ax1.plot(episodes, rewards_history, 'o-', color=colors['fiber'],
             linewidth=1.5, markersize=4, alpha=0.7, label='Episode reward')
    
    # Moving average
    window = min(5, len(rewards_history))
    if len(rewards_history) > window:
        smooth = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
        ax1.plot(episodes[window-1:], smooth, color=colors.get('accent', 'red'),
                 linewidth=2, label=f'Moving avg ({window})')
    
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Reward')
    ax1.set_title('RL Training Progress')
    ax1.legend()
    ax1.grid(True, alpha=0.3, color=colors['grid'])
    
    # Reward distribution
    ax2.hist(rewards_history, bins=15, color=colors['fiber'],
             alpha=0.7, edgecolor=colors['grid'])
    ax2.axvline(np.mean(rewards_history), color='red', linestyle='--', linewidth=2)
    ax2.set_xlabel('Reward')
    ax2.set_ylabel('Count')
    ax2.set_title(f'Reward Distribution\nMean={np.mean(rewards_history):.0f}')
    
    fig.suptitle('RL Training Analysis', color=colors['text'],
                 fontsize=14, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'11_rl_reward_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')

print('\n✓ RL visualization saved (dark + light)')

## 8. Summary / 总结

本教程演示了 FiberNet v4 的完整工作流：

| 步骤 | 内容 | 输出 |
|------|------|------|
| 3.1 | 12种基础单元类型 | `01_gallery_undeformed` |
| 3.2 | 中间点变形演示 | `02_perturbation_comparison` |
| 3.3 | 12种+中间点变形 | `03_gallery_deformed` |
| 3.4 | 批量生成蜂巢变体 | `04_gallery_batch` |
| 4.2 | 变形轨迹(8帧) | `05_trajectory` |
| 4.3 | 应力分布 | `06_stress_distribution` |
| 4.4 | 批量统计 | `07_batch_stats` |
| 5.1 | 特征统计 | `08_feature_stats` |
| 6.1 | ML分析 | `09_ml_analysis` |
| 6.3 | 力-特征相关性 | `10_force_feature_importance` |
| 7.2 | RL奖励曲线 | `11_rl_reward` |

所有可视化都保存了暗色(dark)和亮色(light)两个版本。

**生产参数**:
```python
# N_STRUCTURES = 2000
# PERTURBATION = 0.40
```